In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10, CIFAR100, MNIST, STL10
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mhnlib.utils as mhn_utils
from math import log, sqrt, log10
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path
import re
from einops import rearrange
import pandas as pd
import networkx as nx
import glasbey
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from sklearn.decomposition import PCA
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
#device = torch.device("cpu")

In [7]:
data_folder = Path("paper_results/data/")
DATASET = "stl10"
IS_IMAGE = DATASET in ["mnist", "stl10", "cifar10", "cifar100"]
num_per_label = 5 if IS_IMAGE else None
if num_per_label is not None:
    data_file = data_folder / f"{DATASET}_per_label={num_per_label}.pt"
else:
    data_file = data_folder / f"{DATASET}.pt"
loaded_data = torch.load(data_file)
data = loaded_data["data"]
latents = loaded_data["latents"]
raw_patterns = torch.clone(data.view(data.shape[0],-1)) if latents is None else torch.clone(latents.view(latents.shape[0],-1))
K = raw_patterns.shape[0]
plot_grad = True
plot_pca_1 = True
for is_centered, is_euclidean in [(True, False), (False, True), (False, False)]:
    if is_centered or is_euclidean:
        shift, rms, patterns = mhn_utils.shift_and_rms(raw_patterns)
    else:
        shift = torch.zeros(raw_patterns.shape[-1])
        rms = raw_patterns.norm(dim=1).square().mean().sqrt()
        rms = rms.clamp_min(1e-12)
        patterns = raw_patterns / rms
    gram = patterns @ patterns.T
    if is_euclidean:
        biases = -0.5*torch.sum(patterns**2, dim=-1)
    else:
        biases = torch.zeros(K)
    w_uniform = torch.ones(K)/K
    stab_matr = mhn_utils.get_dual_stability_matrix(gram, w_uniform)
    sm_vals, sm_vecs = torch.linalg.eig(stab_matr)
    sm_vals = sm_vals.real
    largest_sm_val_idx = torch.argmax(sm_vals)
    t_vec = sm_vecs[:, largest_sm_val_idx].real
    beta_c = 1/sm_vals[largest_sm_val_idx].item()

    if plot_pca_1:
        pca_patterns = PCA(n_components=2).fit(patterns)
        pc_max = torch.tensor(pca_patterns.components_[0])
        pc_max_var = torch.tensor(pca_patterns.explained_variance_[0])
        betas = beta_c*torch.logspace(-1, 1, 25)
        #beta_colors_plus = cm.Greens((torch.log10(betas) - torch.log10(betas).min()) / (torch.log10(betas).max() - torch.log10(betas).min()))
        #beta_colors_minus = cm.Oranges((torch.log10(betas) - torch.log10(betas).min()) / (torch.log10(betas).max() - torch.log10(betas).min()))
        epsilons = torch.logspace(-1,0.2, 5)
        x_ics_plus  = patterns.mean(dim=0)[None, :] + epsilons[:, None] * pc_max[None, :] * pc_max_var.sqrt()
        x_ics_minus  = patterns.mean(dim=0)[None, :] - epsilons[:, None] * pc_max[None, :] * pc_max_var.sqrt()
        x_fps_plus = mhn_utils.deterministic_dynamics(patterns.to(device), biases.to(device), betas.to(device), x_ics_plus.to(device), num_iterations=5000, verbose=True).cpu().squeeze(1)
        x_fps_minus = mhn_utils.deterministic_dynamics(patterns.to(device), biases.to(device), betas.to(device), x_ics_minus.to(device), num_iterations=5000, verbose=True).cpu().squeeze(1)
        x_unique_plus = []
        for beta_idx, beta in enumerate(betas):
            x_unique_plus.append(mhn_utils.group_by_distance(x_fps_plus[beta_idx], eps = 1/beta.sqrt().item(), complete_linkage=False)[0])
        x_unique_minus = []
        for beta_idx, beta in enumerate(betas):
            x_unique_minus.append(mhn_utils.group_by_distance(x_fps_minus[beta_idx], eps = 1/beta.sqrt().item(), complete_linkage=False)[0])

        patterns_pca = torch.tensor(pca_patterns.transform(patterns))
        x_ic_plus_pca = torch.tensor(pca_patterns.transform(x_ics_plus))
        x_ic_minus_pca = torch.tensor(pca_patterns.transform(x_ics_minus))

        num_cols = 5
        num_rows = int(np.ceil(len(betas)/num_cols))
        fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols*4, num_rows*4), sharex=True, sharey=True)
        for beta_idx, beta in enumerate(betas):
            row_idx = beta_idx // num_cols
            col_idx = beta_idx % num_cols
            ax = axes[row_idx, col_idx]
            x_plus_pca = torch.tensor(pca_patterns.transform(x_fps_plus[beta_idx]))
            x_minus_pca = torch.tensor(pca_patterns.transform(x_fps_minus[beta_idx]))
            ax.scatter(patterns_pca[:,0], patterns_pca[:,1], c=t_vec, cmap='coolwarm',
                        s=100, vmin=-t_vec.abs().mean(), vmax=t_vec.abs().mean(), edgecolors='black', alpha=0.5)
            ax.scatter(x_plus_pca[:,0], x_plus_pca[:,1], color='darkgreen', s=75)
            ax.scatter(x_minus_pca[:,0], x_minus_pca[:,1], color='orange', s=75)
            ax.quiver(x_ic_plus_pca[:, 0], x_ic_plus_pca[:, 1],
                      x_plus_pca[:, 0] - x_ic_plus_pca[:, 0], x_plus_pca[:, 1] - x_ic_plus_pca[:, 1],
                      angles='xy', scale_units='xy', scale=1, color='darkgreen', alpha=0.5, linewidth=10)
            ax.quiver(x_ic_minus_pca[:, 0], x_ic_minus_pca[:, 1],
                                  x_minus_pca[:, 0] - x_ic_minus_pca[:, 0], x_minus_pca[:, 1] - x_ic_minus_pca[:, 1],
                        angles='xy', scale_units='xy', scale=1, color='orange', alpha=0.5, linewidth=10)   
            ax.set_title(f"$\\beta/\\beta_c={beta/beta_c:.2f}$")
            ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
        pic_filename = f"paper_results/plots_uniform_fp/{data_file.stem}_centered={is_centered}_euclidean={is_euclidean}_pca.pdf"
        plt.savefig(pic_filename, bbox_inches='tight')
        plt.close()

        
        #
        #for epsilon_idx, epsilon in enumerate(epsilons):
        #    x_plus = torch.tensor(pca_patterns.transform(x_fps_plus[:,epsilon_idx]))
        #    x_minus = torch.tensor(pca_patterns.transform(x_fps_minus[:,epsilon_idx]))
        #    plt.plot(x_plus[:,0], x_plus[:,1], color='darkgreen')
        #    plt.plot(x_minus[:,0], x_minus[:,1], color='orange')
        #    break
        #for beta_idx, beta in enumerate(betas):
        #    x_plus = torch.tensor(pca_patterns.transform(x_unique_plus[beta_idx]))
        #    x_minus = torch.tensor(pca_patterns.transform(x_unique_minus[beta_idx]))
        #    plt.scatter(x_plus[:,0], x_plus[:,1], color='darkgreen', s=10, alpha=0.5)
        #    plt.scatter(x_minus[:,0], x_minus[:,1], color='orange', s=10, alpha=0.5)
        


    if plot_grad:
        betas = torch.logspace(log10(1e-5*beta_c), log10(1000*beta_c), 5000, base=np.e)
        linear_term = ((gram @ w_uniform + biases) @ t_vec ).item()
        quadratic_term = 0.5*K*(1/beta_c - 1/betas)
        cubic_term = (K*K/(6*betas))*(t_vec**3).sum()
        epsilon = torch.linspace(-2/K,2/K, 2000)
        functional = linear_term * epsilon
        functional = functional[None,:] + quadratic_term[:, None] * epsilon[None, :]**2
        functional += cubic_term[:, None] * epsilon[None, :]**3
        grad = linear_term + 2*quadratic_term[:, None] * epsilon[None, :] + 3*cubic_term[:, None] * epsilon[None, :]**2
        fig, ax = plt.subplots(figsize=(7,4))
        im = plt.contourf(epsilon, betas, grad, levels=30, cmap='viridis', extend='both')
        plt.contour(epsilon, betas, grad, levels=[0], colors='k', linewidths=2)
        fig.colorbar(im, label=r'$\partial_\epsilon \Phi(\epsilon)$')
        plt.axhline(y=beta_c, color='red', linestyle='--', label=r'$\beta_c$', linewidth=2)
        ax.set_xlabel(r'$\epsilon$')
        ax.set_ylabel(r'$\beta$')
        plt.yscale('log')
        plt.legend()
        pic_filename = f"paper_results/plots_uniform_fp/{data_file.stem}_centered={is_centered}_euclidean={is_euclidean}_grad.pdf"
        plt.savefig(pic_filename, bbox_inches='tight')
        plt.close()

Computing fixed points:   0%|          | 0/5000 [00:00<?, ?it/s]

Computing fixed points:   0%|          | 0/5000 [00:00<?, ?it/s]

Computing fixed points:   0%|          | 0/5000 [00:00<?, ?it/s]

Computing fixed points:   0%|          | 0/5000 [00:00<?, ?it/s]

Computing fixed points:   0%|          | 0/5000 [00:00<?, ?it/s]

Computing fixed points:   0%|          | 0/5000 [00:00<?, ?it/s]